In [ ]:
# Ensure jacopy is importable when this notebook is opened
# directly (not via pytest). Walks up from the notebook's
# directory to the repo root and prepends it to sys.path if
# jacopy isn't already installed into this kernel.
try:
    import jacopy  # noqa: F401
except ModuleNotFoundError:
    import sys
    from pathlib import Path
    here = Path.cwd().resolve()
    for candidate in (here, *here.parents):
        if (candidate / "jacopy" / "__init__.py").is_file():
            sys.path.insert(0, str(candidate))
            break
    import jacopy  # noqa: F401

# 04 — Lie Algebroid

Bu notebook [04_lie_algebroid.md](04_lie_algebroid.md) markdown'ının çalıştırılabilir sürümüdür. `(E, [·,·]_E, ρ)` üçlüsünün `jacopy` içindeki nesneleşmesi, anchor compatibility'nin ayrı aksiyom olarak ele alınışı, ve algebroid Cartan bundle'ına giriş.

## Üçlü: (E, [·,·]_E, ρ)

`LieAlgebroid` bundle adı, section bracket'i, anchor'u ve uyum hedefi olan TM bracket'ini tek objede tutar.

In [ ]:
from jacopy import VectorFields
from jacopy.brackets.lie import LieBracket
from jacopy.calculus.anchor import Anchor
from jacopy.core.expr import Symbol
from jacopy.core.registry import PropertyRegistry
from jacopy.library.lie_algebroid import LieAlgebroid

reg = PropertyRegistry()
E = Symbol("E")
bracket_E = LieBracket(name="[·,·]_E")
rho = Anchor(name="ρ")

A = LieAlgebroid(E, bracket=bracket_E, anchor=rho, name="E-algebroid")
print(A)

## Anchor compatibility — ayrı aksiyom

`ρ([X, Y]_E) = [ρ(X), ρ(Y)]_{TM}` Lie algebroid *tanımı*nın bir parçasıdır; bracket'in kendi aksiyomları içermez. Üç sunumu vardır: obstruction (Expr), koşul (VanishingCondition), ve aksiyom etiketli tek adımlık ProofChain.

In [ ]:
X, Y = VectorFields("X Y", registry=reg)

print('obstruction:')
print(' ', A.anchor_compatibility_obstruction(X, Y, reg))
print()
print('condition:', A.anchor_compatibility_condition(X, Y, reg))
print()
chain = A.prove_anchor_compatibility(X, Y, registry=reg)
print('chain rule:', chain.steps[0].rule)
print('provenance:', chain.steps[0].provenance_tag)
print('justification:', chain.steps[0].justification)

## Algebroid Cartan bundle

Aynı `CartanCalculus` API'si ama `E`-etiketli operatörler ile: `d_E`, `L_{E,X}`, `ι_{E,X}`. Operator-seviyesi `relation()` çağrısı `OperatorEquation` döner. Not: algebroid Cartan magic'i otomatik `verify` ile kapanmıyor (engine cartan-def rewrite'ı TM'ye bağlı); bu beklenen ve kayıtlı. Beş Cartan bağıntısının canlı ispatı için [05_cartan_calculus.md](05_cartan_calculus.md) TM üstünde çalışır.

In [ ]:
cart = A.cartan
print('d:', A.d)
print('L_E,X:', cart.lie_derivative(X))
print('ι_E,X:', cart.interior(X))

eq = cart.relation("cartan_magic", X=X)
print('magic:', eq.lhs, '=', eq.rhs)

## Seeded teorem

Compatibility aksiyomu `theorem_book`'ta kayıtlı — downstream teoremler (algebroid Cartan, Courant–Dorfman köprüsü) tek citation ile bu aksiyomu kullanır.

In [ ]:
from jacopy.library import theorem_book

thm = theorem_book.get("lie_algebroid_anchor_compat")
print('statement:', thm.statement)
print('from_axioms:', thm.from_axioms)

## Sonraki adım

Beş Cartan bağıntısının TM üstünde iki modda canlı ispatı: [05_cartan_calculus.md](05_cartan_calculus.md).